# Implementing Your Own Recommender

This tutorial builds a small Torch recommender in the same three layers used by Compresso Recsys: configuration describes the experiment, an `nn.Module` describes tensor computation, and a trainer connects that computation to sparse recommender data. We then compare the production model with classical and learned baselines.

The model is **Mult-DAE**, the deterministic multinomial denoising autoencoder described alongside Mult-VAE by [Liang et al. (2018)](https://arxiv.org/abs/1802.05814). [AutoRec](https://doi.org/10.1145/2740908.2742726) established autoencoders as collaborative-filtering models; Mult-DAE adapts that family to implicit-feedback ranking with input corruption and a multinomial likelihood.

## MovieLens 1M

We use a fixed strong-generalization user split of MovieLens 1M: models learn from training users and rank held-out movies for test users they have never seen. Ratings of four or five become implicit positive interactions. `eval_draws=1` gives every test user one fold-in history and one target row, keeping the independent unit visible in the statistical analysis. The first run downloads MovieLens and builds the checkpoint; later runs reuse it.

In [1]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix
from torch import nn
from torch.nn import functional as F

from compresso import SRPTensor
import compresso_recsys as cr
from compresso_recsys.evaluation import EvaluationResult, evaluate_recommender
from compresso_recsys.metrics import CalibratedRecall, MRR, NDCG
from compresso_recsys.models import BaseCollaborativeRecommender

In [2]:
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()),
    Path.cwd(),
)
data_checkpoint = project_root / "artifacts/tutorials/ml1m-user-split.zip"
if not data_checkpoint.exists():
    cr.build_recsys_checkpoint(
        dataset="ml1m",
        data_dir=str(project_root / "data"),
        checkpoint_path=str(data_checkpoint),
        split_mode="user_split",
        eval_draws=1,
        seed=42,
        min_entity_text_words=0,
        annotation_source="none",
        show_progress=True,
    )

with cr.read_checkpoint(data_checkpoint) as root:
    split = cr.load_recsys_split(root)

x_train = split["x_train"]
test_source = split["test_source_matrix"]
test_targets = split["test_target_matrix"]
item_ids = split["item_ids"]
test_user_ids = split["test_user_ids"]
x_train.shape, test_source.shape, test_targets.nnz

((4534, 3501), (1000, 3501), 18850)

## Layer 1: configuration

Configuration is data, not runtime state. It records choices needed to repeat or reconstruct the model. Catalog width is deliberately absent: the training data determines it.

In [3]:
@dataclass
class TutorialMultDAEConfig:
    latent_dim: int = 32
    dropout: float = 0.4
    epochs: int = 8
    batch_size: int = 128
    lr: float = 1e-3
    l2_reg: float = 0.01 / 500
    device: str = "cpu"
    seed: int = 0

## Layer 2: the Torch module

The module knows tensors and dimensions. It does not know CSR matrices, item IDs, evaluation metrics, candidate filters, or checkpoints. During training, dropout removes a random part of the normalized history. The network must reconstruct probability mass over the complete history from what remains.

In [4]:
class TutorialMultDAE(nn.Module):
    def __init__(self, n_items, latent_dim, dropout):
        super().__init__()
        self.n_items = n_items
        self.input_dropout = nn.Dropout(dropout)
        self.encoder = nn.Linear(n_items, latent_dim)
        self.decoder = nn.Linear(latent_dim, n_items)

    def forward(self, interactions):
        normalized = F.normalize(interactions, p=2, dim=1)
        corrupted = self.input_dropout(normalized)
        latent = torch.tanh(self.encoder(corrupted))
        return self.decoder(latent)

In [5]:
module = TutorialMultDAE(n_items=x_train.shape[1], latent_dim=32, dropout=0.4)
example = torch.from_numpy(test_source[:4].toarray())
module(example).shape

torch.Size([4, 3501])

## Layer 3: the recommender trainer

`BaseCollaborativeRecommender` supplies streaming `predict`, stable-ID `recommend`, and the common persistence workflow. The concrete trainer owes fitted state, catalog size, `fit`, and `predict_on_batch`. Candidate-local top-k columns must be mapped back to global catalog columns before constructing `SRPTensor`. Seen-item exclusion applies after candidate selection and must fail clearly when fewer than `k` unseen candidates remain.

In [6]:
class TutorialMultDAETrainer(BaseCollaborativeRecommender):
    def __init__(self, config=None):
        self.cfg = config or TutorialMultDAEConfig()
        self.device = torch.device(self.cfg.device)
        self.model = None
        self.optimizer = None
        self._n_items = None

    @property
    def is_fitted(self):
        return self.model is not None

    @property
    def n_items(self):
        return self._n_items

    def fit(self, interactions, *, item_ids=None):
        if not isinstance(interactions, csr_matrix):
            raise TypeError("interactions must be a csr_matrix")
        if interactions.shape[0] == 0 or interactions.shape[1] == 0:
            raise ValueError("interactions must be nonempty")
        if np.any(interactions.data < 0):
            raise ValueError("interactions must be nonnegative")

        torch.manual_seed(self.cfg.seed)
        rng = np.random.default_rng(self.cfg.seed)
        self._n_items = interactions.shape[1]
        self._set_item_ids(item_ids, n_items=self._n_items)
        self.model = TutorialMultDAE(
            self._n_items, self.cfg.latent_dim, self.cfg.dropout
        ).to(self.device)
        self.optimizer = torch.optim.Adam(
            [
                {
                    "params": [self.model.encoder.weight, self.model.decoder.weight],
                    "weight_decay": 2.0 * self.cfg.l2_reg,
                },
                {
                    "params": [self.model.encoder.bias, self.model.decoder.bias],
                    "weight_decay": 0.0,
                },
            ],
            lr=self.cfg.lr,
        )
        active_rows = np.flatnonzero(np.diff(interactions.indptr) > 0)

        for _ in range(self.cfg.epochs):
            self.model.train()
            order = rng.permutation(active_rows)
            for start in range(0, order.size, self.cfg.batch_size):
                rows = order[start : start + self.cfg.batch_size]
                target = torch.from_numpy(
                    interactions[rows].toarray().astype(np.float32)
                ).to(self.device)
                logits = self.model(target)
                loss = -(target * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                self.optimizer.step()
        return self

    def predict_on_batch(
        self, source, *, k, exclude_seen=True, candidate_ids=None
    ):
        source = self._prepare_source(source)
        candidates = self._candidate_rows(candidate_ids)
        if not 1 <= k <= candidates.size:
            raise ValueError(f"k must be in [1, {candidates.size}], got {k}")

        candidate_to_local = np.full(source.shape[1], -1, dtype=np.int64)
        candidate_to_local[candidates] = np.arange(candidates.size)
        seen_counts = np.diff(source.indptr)
        seen_rows = np.repeat(np.arange(source.shape[0]), seen_counts)
        seen_local = candidate_to_local[source.indices]
        selected_seen = seen_local >= 0
        if exclude_seen:
            available = candidates.size - np.bincount(
                seen_rows[selected_seen], minlength=source.shape[0]
            )
            if np.any(available < k):
                raise ValueError("a source row has fewer than k unseen candidates")

        inputs = torch.from_numpy(source.toarray().astype(np.float32)).to(self.device)
        candidate_tensor = torch.as_tensor(candidates, device=self.device)
        self.model.eval()
        with torch.no_grad():
            scores = self.model(inputs)[:, candidate_tensor]
            if exclude_seen:
                scores[
                    torch.as_tensor(seen_rows[selected_seen], device=self.device),
                    torch.as_tensor(seen_local[selected_seen], device=self.device),
                ] = -torch.inf
            values, local_columns = torch.topk(scores, k, dim=1)
            columns = candidate_tensor[local_columns]
        return SRPTensor(cols=columns, vals=values, shape=source.shape)

The tutorial trainer is intentionally compact. A library implementation must additionally validate configuration values, handle empty batches, record history, recreate itself from checkpoint metadata, restore optional optimizer state, and keep device-specific state synchronized. Those details belong to production code and tests, not inside the first readable training loop.

In [7]:
tutorial_model = TutorialMultDAETrainer().fit(x_train, item_ids=item_ids)
tutorial_result = evaluate_recommender(
    tutorial_model,
    source=test_source,
    targets=test_targets,
    metrics=[NDCG(20)],
    sample_ids=test_user_ids,
    collect_per_user=True,
)
dict(tutorial_result)

{'ndcg@20': 0.1738849686086178, 'n_scored_rows': 1000, 'n_units': 1000}

## The production model zoo

Compresso Recsys ships the defended implementation as `MultDAE`, `MultDAEConfig`, and `MultDAETrainer`, plus the variational `MultVAE`, `MultVAEConfig`, and `MultVAETrainer`. The zoo also includes two sanity baselines plus user-user and item-item cosine KNN. KNN fitting uses the optional sklearn dependency (`pip install "compresso-recsys[knn]"`). Checkpoints store typed interaction or similarity matrices and rebuild transient indexes rather than pickling sklearn estimators.

In [8]:
from compresso_recsys.models import (
    EASE,
    EASEConfig,
    ELSAConfig,
    ELSATrainer,
    ItemKNNConfig,
    ItemKNNRecommender,
    MultDAEConfig,
    MultDAETrainer,
    MultVAEConfig,
    MultVAETrainer,
    PopularityBaseline,
    RandomBaseline,
    UserKNNConfig,
    UserKNNRecommender,
)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
TRAINING_SEEDS = tuple(range(5))

fixed_models = {
    "Random": RandomBaseline().fit(x_train, item_ids=item_ids),
    "Popularity": PopularityBaseline().fit(x_train, item_ids=item_ids),
    "UserKNN": UserKNNRecommender(
        UserKNNConfig(n_neighbors=100)
    ).fit(x_train, item_ids=item_ids),
    "ItemKNN": ItemKNNRecommender(
        ItemKNNConfig(n_neighbors=100)
    ).fit(x_train, item_ids=item_ids),
    "EASE": EASE(EASEConfig(l2=100)).fit(x_train, item_ids=item_ids),
}

## One evaluation protocol, several training seeds

Every model sees the same fixed split, histories, targets, candidate catalog, metrics, and user identifiers. Deterministic models are fit once. ELSA, Mult-DAE, and Mult-VAE are each trained from five seeds, because a single SGD run hides initialization and optimization variability. `collect_per_user=True` retains the paired observations required for statistical comparison; aggregate scores alone cannot tell us how variable a difference is across users.

In [9]:
metrics = [CalibratedRecall([10, 20]), NDCG(20), MRR([10, 20])]
fixed_results = {
    name: evaluate_recommender(
        model,
        source=test_source,
        targets=test_targets,
        metrics=metrics,
        sample_ids=test_user_ids,
        collect_per_user=True,
        batch_size=512,
    )
    for name, model in fixed_models.items()
}

seed_results = {"ELSA": [], "Mult-DAE": [], "Mult-VAE": []}
last_mult_dae = None
for seed in TRAINING_SEEDS:
    seeded_models = {
        "ELSA": ELSATrainer(
            ELSAConfig(
                latent_dim=128, epochs=10, batch_size=512, lr=0.05,
                max_output=None, device=device, show_progress=False, seed=seed,
            )
        ).fit(x_train, item_ids=item_ids),
        "Mult-DAE": MultDAETrainer(
            MultDAEConfig(
                latent_dim=128, epochs=20, batch_size=512, dropout=0.5,
                device=device, show_progress=False, seed=seed,
            )
        ).fit(x_train, item_ids=item_ids),
        "Mult-VAE": MultVAETrainer(
            MultVAEConfig(
                latent_dim=128, hidden_dim=256, epochs=20, batch_size=512,
                dropout=0.5, kl_cap=0.2, kl_anneal_steps=200_000,
                device=device, show_progress=False, seed=seed,
            )
        ).fit(x_train, item_ids=item_ids),
    }
    last_mult_dae = seeded_models["Mult-DAE"]
    for name, model in seeded_models.items():
        seed_results[name].append(
            evaluate_recommender(
                model, source=test_source, targets=test_targets, metrics=metrics,
                sample_ids=test_user_ids, collect_per_user=True, batch_size=512,
            )
        )

fixed_summary = pd.DataFrame(
    {name: result.metrics for name, result in fixed_results.items()}
).T
seed_runs = pd.DataFrame(
    [
        {"model": name, "seed": seed, **result.metrics}
        for name, runs in seed_results.items()
        for seed, result in zip(TRAINING_SEEDS, runs, strict=True)
    ]
)
metric_names = list(next(iter(fixed_results.values())).metrics)
seed_summary = seed_runs.groupby("model")[metric_names].agg(["mean", "std"])
display(fixed_summary.round(4))
display(seed_runs.round(4))
seed_summary.round(4)

,calibrated_recall@10,calibrated_recall@20,ndcg@20,mrr@10,mrr@20
Random,0.0082,0.0089,0.0085,0.0259,0.0288
Popularity,0.1589,0.1783,0.1683,0.3304,0.3391
UserKNN,0.2915,0.3077,0.3050,0.5680,0.5720
ItemKNN,0.2102,0.2444,0.2205,0.3921,0.4004
EASE,0.3086,0.3361,0.3250,0.5691,0.5737


,model,seed,calibrated_recall@10,calibrated_recall@20,ndcg@20,mrr@10,mrr@20
0,ELSA,0,0.3283,0.3600,0.3444,0.5855,0.5891
1,ELSA,1,0.3282,0.3607,0.3444,0.5885,0.5933
2,ELSA,2,0.3279,0.3557,0.3405,0.5839,0.5875
3,ELSA,3,0.3291,0.3570,0.3422,0.5853,0.5891
4,ELSA,4,0.3303,0.3584,0.3449,0.5925,0.5970
5,Mult-DAE,0,0.2205,0.2365,0.2343,0.4551,0.4615
6,Mult-DAE,1,0.2205,0.2359,0.2348,0.4562,0.4638
7,Mult-DAE,2,0.2196,0.2361,0.2343,0.4522,0.4593
8,Mult-DAE,3,0.2244,0.2364,0.2345,0.4581,0.4645
9,Mult-DAE,4,0.2276,0.2387,0.2388,0.4604,0.4670


calibrated_recall@10         calibrated_recall@20         ndcg@20  \
                         mean     std                 mean     std    mean   
model                                                                        
ELSA                   0.3288  0.0010               0.3584  0.0021  0.3433   
Mult-DAE               0.2225  0.0034               0.2367  0.0011  0.2353   
Mult-VAE               0.2014  0.0075               0.2158  0.0050  0.2112   

                  mrr@10          mrr@20          
             std    mean     std    mean     std  
model                                             
ELSA      0.0018  0.5871  0.0034  0.5912  0.0039  
Mult-DAE  0.0020  0.4564  0.0031  0.4632  0.0030  
Mult-VAE  0.0067  0.4117  0.0113  0.4196  0.0112

## Paired statistical comparison

The seed table exposes training variability directly. For the paired test, we average each stochastic model's score for each user over its five seeds, then compare those user-level expectations with EASE. Users remain the independent units; seeds are repeated fits of a model on the same split, not extra users. This answers whether the model's average seeded performance differs consistently across users, while the mean/std table answers how much complete training runs vary.

We declare one primary metric, nDCG@20, and compare every model with the same EASE reference in one family. Holm correction controls the family-wise error rate across the seven hypotheses. A positive difference means the candidate outscored EASE.

In [10]:
from compresso_recsys.stats import compare_models

def average_seed_results(runs):
    first = runs[0]
    per_user = {
        metric: np.mean([run.per_user[metric] for run in runs], axis=0)
        for metric in first.per_user
    }
    return EvaluationResult(
        metrics={metric: float(values.mean()) for metric, values in per_user.items()},
        per_user=per_user,
        sample_ids=first.sample_ids,
        n_rows=first.n_rows,
        n_scored_rows=first.n_scored_rows,
        required_k=first.required_k,
        metadata={"training_seeds": list(TRAINING_SEEDS)},
        target_fingerprint=first.target_fingerprint,
    )


comparison_results = dict(fixed_results)
comparison_results.update(
    {name: average_seed_results(runs) for name, runs in seed_results.items()}
)

report = compare_models(
    comparison_results,
    metrics="ndcg@20",
    reference="EASE",
    correction="holm",
    n_resamples=1_999,
    random_state=7,
)
report.to_frame()[
    ["candidate", "difference", "ci_low", "ci_high",
     "adjusted_p_value", "direction"]
].round(4)

,candidate,difference,ci_low,ci_high,adjusted_p_value,direction
0,Random,-0.3165,-0.3279,-0.3052,0.0035,worse
1,Popularity,-0.1567,-0.1673,-0.1459,0.0035,worse
2,UserKNN,-0.0200,-0.0281,-0.0120,0.0035,worse
3,ItemKNN,-0.1045,-0.1151,-0.0944,0.0035,worse
4,ELSA,0.0182,0.0121,0.0245,0.0035,better
5,Mult-DAE,-0.0897,-0.0997,-0.0805,0.0035,worse
6,Mult-VAE,-0.1138,-0.1241,-0.1036,0.0035,worse


Do not reduce this table to significant/not significant. `difference` is the estimated effect, the confidence interval describes uncertainty across test users after seed averaging, and the adjusted p-value answers a separate null-hypothesis question. The seed standard deviations describe a different source of uncertainty: retraining the stochastic model. Five seeds make that variation visible but do not estimate it precisely, and this tutorial is still a worked comparison rather than a tuned leaderboard.

## What changes for other model families

A sequential model inherits `BaseSequentialRecommender`, accepts `ItemSequences`, and must mask the complete history even if its encoder truncates it. A cold-start model inherits `BaseColdStartRecommender`, calls `super().__init__()`, and installs candidate features into its owned mutable catalog. The scoring model changes; the evaluation result, paired statistics, stable-ID recommendation API, and persistence format do not.

## Production persistence

The zoo trainer stores its Torch state, configuration, fitted item IDs, history, and optional optimizer state through the shared safe checkpoint format. Loading defaults to CPU and produces a prediction-ready model.

In [11]:
import tempfile

checkpoint = Path(tempfile.gettempdir()) / "tutorial-mult-dae.zip"
last_mult_dae.save(checkpoint)
restored = MultDAETrainer.load(checkpoint)
history_ids = item_ids[test_source[0].indices[:2]].tolist()
restored.recommend([history_ids], k=5).to_dicts()

[{'260': 2.1207401752471924,
  '1196': 2.0762646198272705,
  '2858': 2.0410895347595215,
  '1198': 2.037564516067505,
  '2571': 2.000257730484009}]